<style>
:root {
  --ink: #17202a;
  --muted: #64748b;
  --paper: #fbf7ef;
  --card: #fffaf0;
  --line: #e7dcc9;
  --navy: #11263f;
  --teal: #0f766e;
  --amber: #b45309;
  --rose: #be123c;
}
body, .jp-Notebook { background: var(--paper); color: var(--ink); }
.jp-RenderedHTMLCommon h1 { color: var(--navy); font-size: 2.2rem; letter-spacing: -0.03em; }
.jp-RenderedHTMLCommon h2 { color: var(--teal); margin-top: 2rem; }
.jp-RenderedHTMLCommon h3 { color: var(--navy); }
.jp-RenderedHTMLCommon p, .jp-RenderedHTMLCommon li { font-size: 1rem; line-height: 1.55; }
.jp-RenderedHTMLCommon code { background: #f1e7d6; color: #7c2d12; padding: 0.12rem 0.28rem; border-radius: 0.25rem; }
.jp-RenderedHTMLCommon table { border-collapse: collapse; width: 100%; background: var(--card); }
.jp-RenderedHTMLCommon th { background: #efe3cd; color: var(--navy); }
.jp-RenderedHTMLCommon th, .jp-RenderedHTMLCommon td { border: 1px solid var(--line); padding: 0.45rem 0.55rem; }
.note-box { background: #fff7ed; border: 1px solid #fed7aa; border-left: 6px solid var(--amber); padding: 0.9rem 1rem; border-radius: 0.65rem; }
.result-box { background: #ecfdf5; border: 1px solid #99f6e4; border-left: 6px solid var(--teal); padding: 0.9rem 1rem; border-radius: 0.65rem; }
</style>


# Rossmann Temporal Validation with Jano

<div class="note-box">
This notebook is the first candidate gold example for Jano. It uses the Rossmann Store Sales dataset to compare a random split, a static chronological holdout and an auditable walk-forward simulation.
</div>

The goal is not to win the Kaggle competition. The goal is to show how Jano helps answer a more operational question:

> Would this model have behaved consistently over time under a specific retraining and evaluation policy?

Dataset source: [Rossmann Store Sales on Kaggle](https://www.kaggle.com/c/rossmann-store-sales). This notebook also supports the public Kaggle dataset mirror [`pratyushakar/rossmann-store-sales`](https://www.kaggle.com/datasets/pratyushakar/rossmann-store-sales), which includes `train.csv`, `test.csv` and `store.csv`.


## 0. Setup

Heavy dataset files stay under `data/raw/rossmann/`, which is ignored by git. If you do not have the files locally, configure the Kaggle CLI first:

```bash
pip install kaggle
mkdir -p ~/.kaggle
# put kaggle.json in ~/.kaggle/ and chmod 600 ~/.kaggle/kaggle.json
```

Then run the download cell below. You can also use Jano's dataset helper:

```bash
python scripts/download_dataset.py rossmann_store_sales --extract
```

If you already have `train.csv` and `store.csv`, place them in `data/raw/rossmann/` and skip the download.


In [1]:
from __future__ import annotations

from pathlib import Path
import subprocess
import sys
import zipfile

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "jano").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from jano import (
    EvaluationProfile,
    PeriodicRetrain,
    TemporalPartitionSpec,
    WalkForwardPolicy,
    WalkForwardRunner,
)

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "rossmann"
TRAIN_CSV = DATA_DIR / "train.csv"
STORE_CSV = DATA_DIR / "store.csv"
RANDOM_SEED = 42


In [2]:
def ensure_rossmann_files() -> bool:
    """Return True when real Rossmann files are available locally."""
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    if TRAIN_CSV.exists() and STORE_CSV.exists():
        print(f"Rossmann files already exist in {DATA_DIR}")
        return True

    command = [
        "kaggle",
        "datasets",
        "download",
        "-d",
        "pratyushakar/rossmann-store-sales",
        "-p",
        str(DATA_DIR),
    ]
    try:
        subprocess.run(command, check=True)
    except (FileNotFoundError, subprocess.CalledProcessError):
        print("Kaggle files are not available in this environment.")
        print("The notebook will use a deterministic Rossmann-like dataset so Jano can still be executed end to end.")
        return False

    for archive in DATA_DIR.glob("*.zip"):
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(DATA_DIR)

    available = TRAIN_CSV.exists() and STORE_CSV.exists()
    if not available:
        print("Downloaded archive did not expose train.csv and store.csv; using synthetic fallback.")
    return available


HAS_REAL_ROSSMANN = ensure_rossmann_files()
DATASET_SOURCE = "Kaggle Rossmann Store Sales" if HAS_REAL_ROSSMANN else "Synthetic Rossmann-like fallback"
print("Dataset source:", DATASET_SOURCE)


Kaggle files are not available in this environment.
The notebook will use a deterministic Rossmann-like dataset so Jano can still be executed end to end.
Dataset source: Synthetic Rossmann-like fallback


## 1. Load and Prepare a Production-Like Frame

Rossmann has one row per store and day. The target is `Sales`. We intentionally avoid `Customers` because it is not available at prediction time in the Kaggle test set, so using it would create a feature-availability problem.

To keep the notebook fast, we use a configurable subset of stores. The temporal structure is preserved: every selected store keeps its complete date history.


In [3]:
def build_synthetic_rossmann_frame(
    *,
    stores: int = 180,
    start: str = "2014-01-01",
    end: str = "2015-07-31",
    seed: int = RANDOM_SEED,
) -> pd.DataFrame:
    """Create a deterministic Rossmann-like daily panel when Kaggle files are unavailable."""
    rng = np.random.default_rng(seed)
    dates = pd.date_range(start, end, freq="D")
    store_ids = np.arange(1, stores + 1)
    grid = pd.MultiIndex.from_product([dates, store_ids], names=["Date", "Store"]).to_frame(index=False)

    store_attrs = pd.DataFrame(
        {
            "Store": store_ids,
            "StoreType": rng.choice(list("abcd"), size=stores, p=[0.45, 0.18, 0.25, 0.12]),
            "Assortment": rng.choice(list("abc"), size=stores, p=[0.50, 0.15, 0.35]),
            "CompetitionDistance": rng.gamma(shape=2.5, scale=900, size=stores).round(0),
        }
    )
    frame = grid.merge(store_attrs, on="Store", how="left")
    frame["DayOfWeek"] = frame["Date"].dt.dayofweek + 1
    frame["Month"] = frame["Date"].dt.month
    frame["WeekOfYear"] = frame["Date"].dt.isocalendar().week.astype(int)
    frame["Promo"] = ((frame["WeekOfYear"] % 4).isin([1, 2]) & (frame["DayOfWeek"] <= 5)).astype(int)
    frame["SchoolHoliday"] = frame["Month"].isin([7, 8, 12]).astype(int)
    frame["StateHoliday"] = np.where(frame["Date"].dt.strftime("%m-%d").isin(["01-01", "12-25"]), "a", "none")
    frame["Open"] = (frame["DayOfWeek"] != 7).astype(int)

    base_by_store = rng.normal(6200, 950, size=stores)
    store_effect = pd.Series(base_by_store, index=store_ids)
    dow_effect = frame["DayOfWeek"].map({1: 350, 2: 250, 3: 150, 4: 100, 5: 450, 6: -700, 7: -2200}).to_numpy()
    promo_effect = frame["Promo"].to_numpy() * rng.normal(1050, 140, size=len(frame))
    seasonal = 550 * np.sin(2 * np.pi * frame["Date"].dt.dayofyear.to_numpy() / 365.25)
    drift = np.where(frame["Date"] >= pd.Timestamp("2015-01-01"), 350, 0)
    noise = rng.normal(0, 420, size=len(frame))
    sales = (
        frame["Store"].map(store_effect).to_numpy()
        + dow_effect
        + promo_effect
        + seasonal
        + drift
        - 0.04 * frame["CompetitionDistance"].to_numpy()
        + noise
    )
    frame["Sales"] = np.where(frame["Open"] == 1, np.maximum(sales, 0), 0).round(0).astype(int)
    return frame.loc[(frame["Open"] == 1) & (frame["Sales"] > 0)].sort_values(["Date", "Store"]).reset_index(drop=True)


def load_rossmann_frame(store_limit: int = 180) -> pd.DataFrame:
    if not HAS_REAL_ROSSMANN:
        return build_synthetic_rossmann_frame(stores=store_limit)

    sales = pd.read_csv(TRAIN_CSV, parse_dates=["Date"], low_memory=False)
    stores = pd.read_csv(STORE_CSV)

    selected_stores = (
        sales.groupby("Store")["Date"]
        .nunique()
        .sort_values(ascending=False)
        .head(store_limit)
        .index
    )
    frame = sales.loc[sales["Store"].isin(selected_stores)].merge(stores, on="Store", how="left")
    frame = frame.loc[(frame["Open"].fillna(1) == 1) & (frame["Sales"] > 0)].copy()
    frame = frame.sort_values(["Date", "Store"]).reset_index(drop=True)

    frame["StateHoliday"] = frame["StateHoliday"].astype(str).replace({"0": "none", "0.0": "none"})
    frame["StoreType"] = frame["StoreType"].fillna("unknown")
    frame["Assortment"] = frame["Assortment"].fillna("unknown")
    frame["CompetitionDistance"] = frame["CompetitionDistance"].fillna(frame["CompetitionDistance"].median())
    frame["Month"] = frame["Date"].dt.month
    frame["WeekOfYear"] = frame["Date"].dt.isocalendar().week.astype(int)

    return frame


rossmann = load_rossmann_frame(store_limit=180)
print("Dataset source:", DATASET_SOURCE)
print(f"rows={len(rossmann):,} stores={rossmann['Store'].nunique():,} date_range={rossmann['Date'].min().date()} -> {rossmann['Date'].max().date()}")
display(rossmann.head())


Dataset source: Synthetic Rossmann-like fallback
rows=89,100 stores=180 date_range=2014-01-01 -> 2015-07-31


,Date,Store,StoreType,Assortment,CompetitionDistance,DayOfWeek,Month,WeekOfYear,Promo,SchoolHoliday,StateHoliday,Open,Sales
0,2014-01-01,1,c,c,1267.0,3,1,1,1,0,a,1,6368
1,2014-01-01,2,a,c,516.0,3,1,1,1,0,a,1,6641
2,2014-01-01,3,c,b,2480.0,3,1,1,1,0,a,1,8358
3,2014-01-01,4,c,a,1287.0,3,1,1,1,0,a,1,6962
4,2014-01-01,5,a,c,3736.0,3,1,1,1,0,a,1,6354


## 2. A Small Model So the Focus Stays on Validation

This notebook uses a simple median model implemented here. It memorizes median sales by store, weekday, promo and store attributes, then falls back to broader medians when a group is unseen.

That keeps the example dependency-light. You can replace this class with any estimator that exposes `fit(X, y)` and `predict(X)`.


In [4]:
FEATURE_COLS = [
    "Store",
    "DayOfWeek",
    "Promo",
    "SchoolHoliday",
    "StateHoliday",
    "StoreType",
    "Assortment",
    "CompetitionDistance",
    "Month",
]


class GroupMedianRegressor:
    def __init__(self, group_cols: list[str], fallback_cols: list[str] | None = None) -> None:
        self.group_cols = list(group_cols)
        self.fallback_cols = list(fallback_cols or [])

    def fit(self, X: pd.DataFrame, y: pd.Series):
        frame = X.copy()
        frame["__target__"] = np.asarray(y, dtype=float)
        self.global_median_ = float(frame["__target__"].median())
        self.group_medians_ = frame.groupby(self.group_cols, dropna=False)["__target__"].median()
        if self.fallback_cols:
            self.fallback_medians_ = frame.groupby(self.fallback_cols, dropna=False)["__target__"].median()
        else:
            self.fallback_medians_ = None
        return self

    def _lookup(self, X: pd.DataFrame, cols: list[str], medians: pd.Series) -> np.ndarray:
        keys = pd.MultiIndex.from_frame(X[cols])
        return medians.reindex(keys).to_numpy(dtype=float)

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        predictions = self._lookup(X, self.group_cols, self.group_medians_)
        missing = np.isnan(predictions)
        if missing.any() and self.fallback_medians_ is not None:
            fallback = self._lookup(X.loc[missing], self.fallback_cols, self.fallback_medians_)
            predictions[missing] = fallback
            missing = np.isnan(predictions)
        predictions[missing] = self.global_median_
        return predictions


def mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))


def rmspe(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denominator = np.maximum(y_true, 1.0)
    return float(np.sqrt(np.mean(((y_true - y_pred) / denominator) ** 2)))


model = GroupMedianRegressor(
    group_cols=["Store", "DayOfWeek", "Promo", "SchoolHoliday", "StateHoliday"],
    fallback_cols=["Store", "DayOfWeek", "Promo"],
)


## 3. Baseline: Random Split

A random split is not automatically wrong. It is wrong when the intended question is temporal: train with the past and evaluate on a future horizon. Here we inspect what the split actually does to the date ranges.


In [5]:
def evaluate_indices(name: str, frame: pd.DataFrame, train_idx: np.ndarray, test_idx: np.ndarray) -> dict[str, object]:
    estimator = GroupMedianRegressor(
        group_cols=["Store", "DayOfWeek", "Promo", "SchoolHoliday", "StateHoliday"],
        fallback_cols=["Store", "DayOfWeek", "Promo"],
    )
    train = frame.iloc[train_idx]
    test = frame.iloc[test_idx]
    estimator.fit(train[FEATURE_COLS], train["Sales"])
    prediction = estimator.predict(test[FEATURE_COLS])
    return {
        "protocol": name,
        "train_rows": len(train),
        "test_rows": len(test),
        "train_start": train["Date"].min(),
        "train_end": train["Date"].max(),
        "test_start": test["Date"].min(),
        "test_end": test["Date"].max(),
        "mae": mae(test["Sales"], prediction),
        "rmspe": rmspe(test["Sales"], prediction),
    }


rng = np.random.default_rng(RANDOM_SEED)
indices = rng.permutation(len(rossmann))
cut = int(len(indices) * 0.8)
random_result = evaluate_indices("random_80_20", rossmann, indices[:cut], indices[cut:])
display(pd.DataFrame([random_result]))


,protocol,train_rows,test_rows,train_start,train_end,test_start,test_end,mae,rmspe
0,random_80_20,71280,17820,2014-01-01,2015-07-31,2014-01-01,2015-07-31,498.16748,0.102699


## 4. Baseline: Static Chronological Holdout

A chronological holdout is a better operational baseline: train on the past and test on the last six weeks. It still gives only one snapshot. It cannot show whether performance was stable across multiple deployment dates.


In [6]:
holdout_days = 42
holdout_start = rossmann["Date"].max() - pd.Timedelta(days=holdout_days - 1)
holdout_train_idx = rossmann.index[rossmann["Date"] < holdout_start].to_numpy()
holdout_test_idx = rossmann.index[rossmann["Date"] >= holdout_start].to_numpy()

baseline_comparison = pd.DataFrame([
    random_result,
    evaluate_indices("last_42_days_holdout", rossmann, holdout_train_idx, holdout_test_idx),
])
display(baseline_comparison)

random_temporal_overlap = (
    baseline_comparison.loc[0, "train_end"] >= baseline_comparison.loc[0, "test_start"]
    and baseline_comparison.loc[0, "test_end"] >= baseline_comparison.loc[0, "train_start"]
)
print("Interpretation:")
print(f"- Random split train/test date ranges overlap: {random_temporal_overlap}")
print("- The chronological holdout answers a cleaner production-like question, but only for one future window.")


Interpretation:
- Random split train/test date ranges overlap: True
- The chronological holdout answers a cleaner production-like question, but only for one future window.


,protocol,train_rows,test_rows,train_start,train_end,test_start,test_end,mae,rmspe
0,random_80_20,71280,17820,2014-01-01,2015-07-31,2014-01-01,2015-07-31,498.167480,0.102699
1,last_42_days_holdout,82620,6480,2014-01-01,2015-06-19,2015-06-20,2015-07-31,486.530015,0.087664


## 5. Plan the Walk-Forward Simulation Before Training

This is where Jano enters. [`plan()`](https://marmurar.github.io/jano/concepts.html#planning-before-materialization) precomputes the fold geometry before materializing train/test slices or fitting a model.

Here the policy means:

- use 180 days of training history;
- evaluate the next 42 days;
- move forward by 42 days;
- align windows to calendar days;
- keep at most eight folds for a compact notebook.


In [7]:
policy = WalkForwardPolicy(
    time_col="Date",
    partition=TemporalPartitionSpec(
        layout="train_test",
        train_size="180D",
        test_size="42D",
        calendar_frequency="D",
    ),
    step="42D",
    strategy="rolling",
    start_at="2014-01-01",
    max_folds=8,
)

plan = policy.plan(rossmann, title="Rossmann 180D train / 42D test walk-forward plan")
plan_df = plan.to_frame()
display(plan_df[["iteration", "fold", "train_start", "train_end", "train_rows", "test_start", "test_end", "test_rows"]])


,iteration,fold,train_start,train_end,train_rows,test_start,test_end,test_rows
0,0,0,2014-01-01,2014-06-30,27720,2014-06-30,2014-08-11,6480
1,1,1,2014-02-12,2014-08-11,27720,2014-08-11,2014-09-22,6480
2,2,2,2014-03-26,2014-09-22,27720,2014-09-22,2014-11-03,6480
3,3,3,2014-05-07,2014-11-03,27720,2014-11-03,2014-12-15,6480
4,4,4,2014-06-18,2014-12-15,27720,2014-12-15,2015-01-26,6480
5,5,5,2014-07-30,2015-01-26,27720,2015-01-26,2015-03-09,6480
6,6,6,2014-09-10,2015-03-09,27720,2015-03-09,2015-04-20,6480
7,7,7,2014-10-22,2015-04-20,27720,2015-04-20,2015-06-01,6480


## 6. Run the Same Model Across the Planned Folds

[`WalkForwardRunner`](https://marmurar.github.io/jano/api.html#jano.runner.WalkForwardRunner) owns the fit/predict/metric loop. The splitter still owns temporal geometry.

This keeps the experiment explicit: same model, same features, same metric, multiple temporal deployment dates.


In [8]:
evaluation = EvaluationProfile(
    metrics={"mae": mae, "rmspe": rmspe},
    metric_directions={"mae": "min", "rmspe": "min"},
    primary_metric="rmspe",
)

runner = WalkForwardRunner(
    model=model,
    target_col="Sales",
    feature_cols=FEATURE_COLS,
    retrain="always",
    evaluation=evaluation,
)

run = runner.run(policy, rossmann)
fold_metrics = run.to_frame()
display(fold_metrics)
display(pd.DataFrame([run.summary()]))

print("Interpretation:")
print("- Each row is one simulated deployment date.")
print("- The metric trajectory shows whether the same model strategy behaves consistently across time.")
print("- This is information a single random split or a single holdout cannot expose.")


Interpretation:
- Each row is one simulated deployment date.
- The metric trajectory shows whether the same model strategy behaves consistently across time.
- This is information a single random split or a single holdout cannot expose.


,fold,retrained,last_retrain_fold,train_rows,test_rows,train_start,train_end,test_start,test_end,mae,rmspe
0,0,True,0,27720,6480,2014-01-01,2014-06-30,2014-06-30,2014-08-11,595.588735,0.130740
1,1,True,1,27720,6480,2014-02-12,2014-08-11,2014-08-11,2014-09-22,691.332022,0.151054
2,2,True,2,27720,6480,2014-03-26,2014-09-22,2014-09-22,2014-11-03,815.460185,0.178641
3,3,True,3,27720,6480,2014-05-07,2014-11-03,2014-11-03,2014-12-15,406.995679,0.087519
4,4,True,4,27720,6480,2014-06-18,2014-12-15,2014-12-15,2015-01-26,703.390664,0.123267
5,5,True,5,27720,6480,2014-07-30,2015-01-26,2015-01-26,2015-03-09,1115.937346,0.161852
6,6,True,6,27720,6480,2014-09-10,2015-03-09,2015-03-09,2015-04-20,946.986265,0.143009
7,7,True,7,27720,6480,2014-10-22,2015-04-20,2015-04-20,2015-06-01,423.468981,0.070225


,folds,retrain_policy,retrain_events,retrain_ratio,metrics,primary_metric,mae_mean,mae_best,mae_best_fold,rmspe_mean,rmspe_best,rmspe_best_fold
0,8,AlwaysRetrain,8,1.0,"[mae, rmspe]",rmspe,712.394985,406.995679,3,0.130788,0.070225,7


## 7. Compare Retraining Policies

The same fold geometry can answer a second operational question: do we need to retrain every fold, or is a cheaper cadence enough for this model and dataset?


In [9]:
policy_runs = {}
for label, kwargs in {
    "always": {"retrain": "always"},
    "never": {"retrain": "never"},
    "periodic_2": {"retrain_policy": PeriodicRetrain(2)},
}.items():
    runner = WalkForwardRunner(
        model=model,
        target_col="Sales",
        feature_cols=FEATURE_COLS,
        evaluation=evaluation,
        **kwargs,
    )
    policy_runs[label] = runner.run(policy, rossmann)

comparison = pd.DataFrame(
    {label: result.summary() for label, result in policy_runs.items()}
).T.reset_index(names="policy")
display(comparison[["policy", "folds", "retrain_events", "rmspe_mean", "rmspe_best", "rmspe_best_fold", "mae_mean"]])

best_policy = comparison.sort_values("rmspe_mean").iloc[0]
print("Interpretation:")
print(f"- Best mean RMSPE in this run: {best_policy['policy']} ({best_policy['rmspe_mean']:.4f}).")
print("- The important point is not that one policy always wins; Jano makes the policy comparison explicit and reproducible.")


Interpretation:
- Best mean RMSPE in this run: never (0.1230).
- The important point is not that one policy always wins; Jano makes the policy comparison explicit and reproducible.


,policy,folds,retrain_events,rmspe_mean,rmspe_best,rmspe_best_fold,mae_mean
0,always,8,8,0.130788,0.070225,7,712.394985
1,never,8,1,0.123012,0.079614,5,630.880469
2,periodic_2,8,4,0.145853,0.122938,7,807.117564


In [10]:
trajectory = pd.concat(
    [result.metric_trajectory().assign(policy=label) for label, result in policy_runs.items()],
    ignore_index=True,
)
rmspe_trajectory = trajectory.loc[trajectory["metric"] == "rmspe"]
rmspe_pivot = rmspe_trajectory.pivot(index="fold", columns="policy", values="value")
display(rmspe_pivot)

print("Interpretation:")
print("- This table is plot-ready data: one fold per row, one retraining policy per column.")
print("- Downstream tools can render it as a line chart, dashboard card or report without Jano owning presentation style.")


Interpretation:
- This table is plot-ready data: one fold per row, one retraining policy per column.
- Downstream tools can render it as a line chart, dashboard card or report without Jano owning presentation style.


policy,always,never,periodic_2
fold,,,
0,0.130740,0.130740,0.130740
1,0.151054,0.172087,0.172087
2,0.178641,0.191745,0.178641
3,0.087519,0.147378,0.127249
4,0.123267,0.089010,0.123267
5,0.161852,0.079614,0.168895
6,0.143009,0.092673,0.143009
7,0.070225,0.080848,0.122938


## 8. What Jano Adds Here

<div class="result-box">
Jano does not claim that random split is always invalid. It makes the temporal assumption testable. If random, holdout and walk-forward behavior are consistent, that is useful evidence. If they diverge, Jano shows when, where and under which retraining policy.
</div>

In this example, the important artifacts are:

- `plan_df`: the auditable temporal geometry before training;
- `fold_metrics`: fold-by-fold behavior under one policy;
- `comparison`: aggregate comparison of retraining policies;
- `trajectory`: plot-ready metric data for custom reports, dashboards or agents.

This is the core paper/landing-page argument: temporal validation should be an empirical simulation, not a hidden assumption inside a single split.
